In [1]:
import datetime

# Obtener la fecha actual
hoy = datetime.date.today()

# Calcular el primer día del mes actual y restar un día para obtener el último día del mes anterior
primer_dia_mes_actual = hoy.replace(day=1)
ultimo_dia_mes_anterior = primer_dia_mes_actual - datetime.timedelta(days=1)

# Extraer mes y año del mes anterior
mes_anterior = ultimo_dia_mes_anterior.month-1
anio_anterior = ultimo_dia_mes_anterior.year

# Asignar variables
MES_ACTUAL_EN_RELIQUIDACION = f"'{anio_anterior}{mes_anterior:02d}'"

# Imprimir resultados
print("Mes reliquidar =", MES_ACTUAL_EN_RELIQUIDACION)


Mes reliquidar = '202609'


In [2]:
#Importacion de librerias
import requests
import pandas as pd
from unidecode import unidecode
import numpy as np
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker

In [3]:
# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print("Error:", e)

Conexion a la base de datos realizada


In [4]:
#CONSULTA PARA CREAR LA TABLA TEMPORAL EN SQL 
with engine.connect() as connection:
    try:

        sql_query = text(f'''
         IF object_id('CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_BASE_RELIQUIDACION') IS NOT NULL
            DROP TABLE CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_BASE_RELIQUIDACION
        Select t1.Doc_Estudiante,t1.PERIODO,t1.Documento_del_Asesor,t1.Propietario_del_registro,t1.Programa_vendido,t1.Correo_Asesor,t1.Periodo_Pago
            ,t3.EST_MATRICULADO,t3.NCR_ANULACION,t3.TIPO,t3.ORDEN,
            t3.VAL_LIQUIDADO,t3.NUEVO,t3.ESTADO_PAGO,t3.Doc_Alum as Documento_Estudiante_zoho
        INTO CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_BASE_RELIQUIDACION
        from Comisiones_Contact.Etiquetas t1
                Left join (Select * from (
        select Doc_Alum,ESTADO_PAGO,NCR_ANULACION,NUEVO,EST_MATRICULADO,
                PERIODO,ORDEN,VAL_LIQUIDADO,TIPO
                ,row_number() over (partition by Doc_Alum,PERIODO  order by ESTADO_PAGO desc,NCR_ANULACION desc, EST_MATRICULADO,ORDEN asc ) cont
                            from ##Ventas where DOCUMENTO in ( 'FAMA','FECO')  and PERIODO not like '%I%'
                                                                            /* and EST_MATRICULADO = '1-Activo'
                                                                                and NUEVO = 'NUEVO'
                                                                                AND NCR_ANULACION = 0
            AND ESTADO_PAGO = 'PAGO'*/)t1 where cont = 1) t3 on t1.Doc_Estudiante = t3.Doc_Alum and t1.PERIODO = t3.PERIODO
                where Periodo_Pago = {MES_ACTUAL_EN_RELIQUIDACION} and EstadoFinal <> 'ok'
        ''')
        connection.execute(sql_query)
        connection.commit()  # Asegurar que los cambios se guarden en la base de datos
        print("Consulta realizada con exito")
    except Exception as e:
        print(f"Error al eliminar registros: {e}")
    finally:
        connection.close()

Consulta realizada con exito


In [5]:
#CONSULTA PARA CREAR LA TABLA TEMPORAL EN SQL 
with engine.connect() as connection:
    try:

        sql_query = text(f'''
                 Delete from Comisiones_Contact.EtiquetasReliquidacion where Periodo_Pago = {MES_ACTUAL_EN_RELIQUIDACION} 

                INSERT INTO Comisiones_Contact.EtiquetasReliquidacion
                select *,
                    case
                            when Motivo =  'OK' THEN 'OK'
                ELSE Motivo END AS EstadoFinal
                    from (select distinct t1.*,
                                            case
                                                when t1.Documento_Estudiante_zoho is null
                                                    then 'No coincide con Iceberg o Sinu'
                                                when t1.ESTADO_PAGO <> 'PAGO' then 'No Registra Pago'
                                                when t1.ESTADO_PAGO IS NULL then 'No Registra Pago'
                                                when t1.EST_MATRICULADO IS NULL then 'No Registra Matricula'
                                                --  when t1.NCR_ANULACION < 0 then 'Tiene Anulación'
                                                when t1.NUEVO = 'ANTIGUO' and TIPO <> 'Educacion continuada' then 'Estudiante ANTIGUO'
                                                when t1.EST_MATRICULADO <> '1-Activo'
                                                    then 'Estado Diferente al Activo ' + t1.EST_MATRICULADO
                                                when t1.VAL_LIQUIDADO < 200000 then 'Valor liquidado Menor 20%'
                                                else 'ok' end Motivo,
                                            t2.Propietario_de_Fase_Name--,t2.Correo_electronico_asesor
                            from CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_BASE_RELIQUIDACION t1
                                        left join CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_ZOHO_FUNELL t2
                                                on t1.Doc_Estudiante = t2.Número_de_Documento and t1.PERIODO = t2.Periodo and
                                                    t1.Correo_Asesor = t2.Propietario_de_Fase_Name
                            )t1
        ''')
        connection.execute(sql_query)
        connection.commit()  # Asegurar que los cambios se guarden en la base de datos
        print("Consulta realizada con exito")
    except Exception as e:
        print(f"Error al eliminar registros: {e}")
    finally:
        connection.close()

Consulta realizada con exito


In [6]:
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'

try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    #archivo_orig = pd.read_sql("select * from CUN_REPOSITORIO.Comisiones.Com_Telec_Oct_2024_2_yCT", engine2_1)
    Etiquetas_penultimo_paso = pd.read_sql(f'''
        select * from CUN_REPOSITORIO.COMISIONES_CONTACT.EtiquetasReliquidacion
                                           WHERE Periodo_Pago = {MES_ACTUAL_EN_RELIQUIDACION}  and EstadoFinal ='OK'
                               ''', engine2_1)
    Consulta_z2 = Etiquetas_penultimo_paso.copy()
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")
#-------------------------------------------------------------------------------------------
Etiquetas_penultimo_paso['Doc_Estudiante'] = Etiquetas_penultimo_paso['Doc_Estudiante'].astype(int).astype(str)
Etiquetas_penultimo_paso['PERIODO'] = Etiquetas_penultimo_paso['PERIODO'].astype(str)
Etiquetas_penultimo_paso['Llave_per'] = Etiquetas_penultimo_paso['Doc_Estudiante'] + Etiquetas_penultimo_paso['PERIODO']
#--------------------------------------------------------------------------------------------
Etiquetas_penultimo_paso.shape
#--------------------------------------------------------------------------------------------
#--------------------------------------------------------------------------------------------

Conexion a la base de datos realizada
Conexión cerrada


(49, 19)

In [7]:
#Eliminamos los registros de la tabla para luego insertarlos nuevamente
with engine.connect() as connection:
    try:

        sql_query = text(f'''
                DELETE FROM CUN_REPOSITORIO.COMISIONES_CONTACT.EtiquetasReliquidacion WHERE Periodo_Pago = {MES_ACTUAL_EN_RELIQUIDACION} and EstadoFinal ='OK'
        ''')
        connection.execute(sql_query)
        connection.commit()  # Asegurar que los cambios se guarden en la base de datos
        print("Consulta realizada con exito")
    except Exception as e:
        print(f"Error al eliminar registros: {e}")
    finally:
        connection.close()

Consulta realizada con exito


# CONSULTA TOTAL y tratamiento general

In [8]:
##para_orden = Etiquetas_penultimo_paso['Doc_Estudiante'].unique()
##valores_unicos_df1_orden = "', '".join(map(str, para_orden))
##valores_unicos_df1_orden = f"'{valores_unicos_df1_orden}'"
###----------------------------------------------------------------------------------
##host2_1 = '172.16.1.33'
##port2_1 = '1433'
##database2_1 = 'CUN_REPOSITORIO'
##
##try:
##    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
##    engine2_1 = create_engine(url2_1) 
##    print("Conexion a la base de datos realizada")
##    df_ordenes= pd.read_sql(f'''
##SELECT DISTINCT DOC_ALUM AS  Documento_Estudiante_zoho, 
##    PERIODO, 
##    COD_UNI,
##    FUERZAZOHO AS FuerzaComercialFinal,
##    NUEVO,
##    CLASE_ACTUAL
##FROM ZOHO.BASE_PERSONAS
##     WHERE Documento_Estudiante_zoho IN ({valores_unicos_df1_orden})
##       AND ESTADO_PAGO = 'PAGO'
##       AND EST_MATRICULADO = '1-Activo'
##                            ''', engine2_1)
##    df_ordenes2 = df_ordenes.copy()
##        
##except Exception as e:
##    print("Error:", e)
##finally:
##    # Cerrar la conexión
##    if 'engine2_1' in locals():
##        engine2_1.dispose()
##        print("Conexión cerrada")
##df_ordenes['llave_orden_sql']=df_ordenes['Documento_Estudiante_zoho'].astype(str) + df_ordenes['PERIODO'].astype(str)
##df_ordenes['llave_total_cod_uni']=df_ordenes['Documento_Estudiante_zoho'].astype(str) + df_ordenes['COD_UNI'].astype(str)

In [9]:
# Extrae datos de la tabla Base_Personas en la BD CUN_REPOSITORIO
# usando los documentos de Etiquetas_penultimo_paso

# Obtiene los documentos únicos
para_orden = Etiquetas_penultimo_paso['Doc_Estudiante'].unique()
valores_unicos_df1_orden = "', '".join(map(str, para_orden))
valores_unicos_df1_orden = f"'{valores_unicos_df1_orden}'"

#----------------------------------------------------------------------------------
# Parámetros de conexión
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'

try:
    # Crea la conexión
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(
        host=host2_1, 
        port=port2_1, 
        db=database2_1
    )
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")

    # Ejecuta la consulta
    df_ordenes = pd.read_sql(f"""
        SELECT DISTINCT 
            DOC_ALUM AS Documento_Estudiante_zoho, 
            PERIODO, 
            COD_UNI,
            FUERZAZOHO AS FuerzaComercialFinal,
            NUEVO,
            CLASE_ACTUAL
        FROM ZOHO.Base_Personas
        WHERE DOC_ALUM IN ({valores_unicos_df1_orden})
          AND ESTADO_PAGO = 'PAGO'
          AND EST_MATRICULADO = '1-Activo'
    """, engine2_1)

    # Copia del dataframe (opcional)
    df_ordenes2 = df_ordenes.copy()

except Exception as e:
    print("Error:", e)

finally:
    # Cierra la conexión
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")

#----------------------------------------------------------------------------------
# Crea las llaves concatenadas
df_ordenes['llave_orden_sql'] = (
    df_ordenes['Documento_Estudiante_zoho'].astype(str) 
    + df_ordenes['PERIODO'].astype(str)
)

df_ordenes['llave_total_cod_uni'] = (
    df_ordenes['Documento_Estudiante_zoho'].astype(str) 
    + df_ordenes['COD_UNI'].astype(str)
)

Conexion a la base de datos realizada
Conexión cerrada


In [10]:
#ARMO UNA MINI CONSULTA PERO SOLO PARA VALIDAR COD_UNI EN EL PERIODO ACTUAL
df_recibos_per_Actual=df_ordenes[df_ordenes['llave_orden_sql'].isin(Etiquetas_penultimo_paso['Llave_per'])]
df_recibos_per_Actual=df_recibos_per_Actual.drop_duplicates(subset='llave_orden_sql', keep='first')
#-----------------EXPORTANDO EVIDENCIA--------------------------------------
from datetime import datetime
fecha_hoy_rc = datetime.now().strftime('%Y-%m-%d')
df_recibos_per_Actual.to_excel(f'Evidencia_Recibos_cont_per_actual{fecha_hoy_rc}_Reliq.xlsx')
#-------------------------------


Me traigo el codigo de unidad a mi liquidacion actual

In [11]:
Etiquetas_penultimo_paso = Etiquetas_penultimo_paso.merge(df_recibos_per_Actual[['llave_orden_sql', 'COD_UNI']], 
                                                left_on='Llave_per', 
                                                right_on='llave_orden_sql', 
                                                how='left')
Etiquetas_penultimo_paso.drop(columns=['llave_orden_sql'], inplace=True)
Etiquetas_penultimo_paso.shape

(49, 20)

# VALIDACION VS HISTORICO TELECAMPUS

In [12]:
para_orden = Etiquetas_penultimo_paso['Doc_Estudiante'].unique()
valores_unicos_df1_orden = "', '".join(map(str, para_orden))
valores_unicos_df1_orden = f"'{valores_unicos_df1_orden}'"
#----------------------------------------------------------------------------------
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'

try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    df_histo_telec= pd.read_sql(f'''
SELECT DISTINCT DOC_ALUM, PERIODO, COD_UNI FROM CUN_REPOSITORIO.FINANCIERA.HISTORICO_COM_TELEC_2025
     WHERE DOC_ALUM IN ({valores_unicos_df1_orden})
                            ''', engine2_1)
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")
df_histo_telec['llave_hist_tel_cod_uni']=df_histo_telec['DOC_ALUM'].astype(str) + df_histo_telec['COD_UNI'].astype(str)
df_histo_telec['llave_hist_CC_PER']=df_histo_telec['DOC_ALUM'].astype(str) + df_histo_telec['PERIODO'].astype(str)

df_histo_telec
#--------------------------------
#--------------------------------

Conexion a la base de datos realizada
Conexión cerrada


,DOC_ALUM,PERIODO,COD_UNI,llave_hist_tel_cod_uni,llave_hist_CC_PER
0,1019035033,25V02,PVC06,1019035033PVC06,101903503325V02
1,1024567467,26V01,VTC06,1024567467VTC06,102456746726V01


Validacion VS TELECAMPUS directo POR PERIODO

In [13]:
Etiquetas_penultimo_paso['Esta_en_hist_telec_x_per']=Etiquetas_penultimo_paso['Llave_per'].isin(df_histo_telec['llave_hist_CC_PER']).astype(int)
Etiquetas_penultimo_paso.loc[Etiquetas_penultimo_paso['Esta_en_hist_telec_x_per']==1, 'EstadoFinal'] = 'ENTRO CON OTRA FUERZA_T'
#----------------------------------

Validacion VS TELECAMPUS POR CODIGO DE UNIDAD

In [14]:
#ARMO LLAVE POR COD UNI PARA TODOS
Etiquetas_penultimo_paso['Llave_liq_cod_uni']=Etiquetas_penultimo_paso['Doc_Estudiante'] + Etiquetas_penultimo_paso['COD_UNI']

"Divido mi liquidacion entre los que si obtuvieron su cod_uni y los que no"
Etiquetas_penultimo_paso_sin_cod_uni=Etiquetas_penultimo_paso[Etiquetas_penultimo_paso['COD_UNI'].isna()]
Etiquetas_penultimo_Con_cod_uni=Etiquetas_penultimo_paso[Etiquetas_penultimo_paso['COD_UNI'].notna()]

In [15]:
condicion_histor_telec_novedades=(
    (df_histo_telec['COD_UNI'].isna())|
    (df_histo_telec['COD_UNI']=='0')|
    (df_histo_telec['COD_UNI']==0)|
    (df_histo_telec['COD_UNI']=='Null')|
    (df_histo_telec['COD_UNI']=='null')|
    (df_histo_telec['COD_UNI']=='NULL')|
    (df_histo_telec['COD_UNI']=='')|
    (df_histo_telec['COD_UNI']==' ')|
    (df_histo_telec['COD_UNI']=='na')    
)

#divido mi historico telecampus en dos, que tiene COD_UNI Valido y los que no
df_histo_telec_SOLO_NOvedades=df_histo_telec[condicion_histor_telec_novedades]
df_histo_telec_Depurado=df_histo_telec[~condicion_histor_telec_novedades]

#ETIQUETEO MI LIQUIDACION CON COD_UNI
# VALIDO CONTRA EL HISTORICO telec de novedades por cc y contra el historico tele depurado por llave
Etiquetas_penultimo_Con_cod_uni['Esta_En_histor_telec_depur']=Etiquetas_penultimo_Con_cod_uni['Llave_liq_cod_uni'].isin(df_histo_telec_Depurado['llave_hist_tel_cod_uni']).astype(int)
Etiquetas_penultimo_Con_cod_uni['Esta_En_histor_telec_NOved']=Etiquetas_penultimo_Con_cod_uni['Doc_Estudiante'].isin(df_histo_telec_SOLO_NOvedades['DOC_ALUM']).astype(int)

condicion_cod_uni_liq=((Etiquetas_penultimo_Con_cod_uni['Esta_En_histor_telec_depur']==1)|(Etiquetas_penultimo_Con_cod_uni['Esta_En_histor_telec_NOved']==1))
Etiquetas_penultimo_Con_cod_uni.loc[condicion_cod_uni_liq, 'EstadoFinal'] = 'ENTRO CON OTRA FUERZA_T'

#ETIQUETO LOS QUE NO TIENEN COD UNI
Etiquetas_penultimo_paso_sin_cod_uni['Esta_En_histor_telec_TOTAL']=Etiquetas_penultimo_paso_sin_cod_uni['Doc_Estudiante'].isin(df_histo_telec['DOC_ALUM']).astype(int)
Etiquetas_penultimo_paso_sin_cod_uni.loc[Etiquetas_penultimo_paso_sin_cod_uni['Esta_En_histor_telec_TOTAL']==1, 'EstadoFinal'] = 'ENTRO CON OTRA FUERZA_T'

#VUELVO A UNIR MI LIQUIDACION
Etiquetas_penultimo_paso=[]
Etiquetas_penultimo_paso = pd.concat([Etiquetas_penultimo_Con_cod_uni, Etiquetas_penultimo_paso_sin_cod_uni], ignore_index=True)
#--------------------------------------------
#--------------------------------------------

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_6840\415972995.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Etiquetas_penultimo_Con_cod_uni['Esta_En_histor_telec_depur']=Etiquetas_penultimo_Con_cod_uni['Llave_liq_cod_uni'].isin(df_histo_telec_Depurado['llave_hist_tel_cod_uni']).astype(int)
C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_6840\415972995.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Etiquetas_penultimo_Con_cod_uni['Esta_En_histor_telec_NOved']=Etiquetas_penul

# COMIENZA LA VALIDACION DE NUEVOS - POR PERIODO ACTUAL

In [16]:
# CREO UNA CONSULTA CON SOLO DONDE NUEVO = NUEVO Y CLASE ACTUAL= NUEVO
Consulta_recibos_per_Actual_solo_nuevos=df_recibos_per_Actual[(df_recibos_per_Actual['NUEVO']=='NUEVO')&(df_recibos_per_Actual['CLASE_ACTUAL']=='Nuevo')]
Consulta_recibos_per_Actual_solo_nuevos.shape

#AHORA SIMPLEMENTE COMPARO SI MIS LIQUIDACION ACTUAL EN SU RESPECTIVO PERIODO SI ES NUEVO
Etiquetas_penultimo_paso['Esta_en_consulta_Nuevos'] =Etiquetas_penultimo_paso['Llave_per'].isin(Consulta_recibos_per_Actual_solo_nuevos['llave_orden_sql']).astype(int)
Etiquetas_penultimo_paso['Esta_en_consulta_per_Actual'] =Etiquetas_penultimo_paso['Llave_per'].isin(df_recibos_per_Actual['llave_orden_sql']).astype(int)

condicion_real_antiguo=((Etiquetas_penultimo_paso['Esta_en_consulta_Nuevos']==0) & (Etiquetas_penultimo_paso['Esta_en_consulta_per_Actual']==1))
Etiquetas_penultimo_paso.loc[condicion_real_antiguo, 'EstadoFinal'] = 'Estudiante ANTIGUO_PER_ACT'

# VALIDACION FUERZA DE VENTA POR PERIODO ACTUAL

In [17]:
# CREO UNA CONSULTA DONDE HAYAN FUERZAS DE VENTA VALIDAS Y LUEGO DONDE SOLO SEA EL CONTACT
Consulta_recibos_per_Actual_varias_fuerzas=df_recibos_per_Actual.copy()
Consulta_recibos_per_Actual_varias_fuerzas=Consulta_recibos_per_Actual_varias_fuerzas[Consulta_recibos_per_Actual_varias_fuerzas['FuerzaComercialFinal']!= 'N/A']
Consulta_recibos_per_Actual_varias_fuerzas=Consulta_recibos_per_Actual_varias_fuerzas[Consulta_recibos_per_Actual_varias_fuerzas['FuerzaComercialFinal']!= 'NA']
Consulta_recibos_per_Actual_varias_fuerzas=Consulta_recibos_per_Actual_varias_fuerzas[Consulta_recibos_per_Actual_varias_fuerzas['FuerzaComercialFinal']!= 'PENDIENTE']
Consulta_recibos_per_Actual_varias_fuerzas=Consulta_recibos_per_Actual_varias_fuerzas[Consulta_recibos_per_Actual_varias_fuerzas['FuerzaComercialFinal'].notna()]

#Creo df solo con CONTACT
Consulta_recibos_per_Actual_solo_CONTACT=Consulta_recibos_per_Actual_varias_fuerzas[(Consulta_recibos_per_Actual_varias_fuerzas['FuerzaComercialFinal']=='CONTACT')|(Consulta_recibos_per_Actual_varias_fuerzas['FuerzaComercialFinal']=='CONTACT ')]

#AHORA SIMPLEMENTE COMPARO SI MIS LIQUIDACION ACTUAL EN SU RESPECTIVO PERIODO SI ES NUEVO
Etiquetas_penultimo_paso['Esta_en_consulta_CONTACT'] =Etiquetas_penultimo_paso['Llave_per'].isin(Consulta_recibos_per_Actual_solo_CONTACT['llave_orden_sql']).astype(int)
Etiquetas_penultimo_paso['Esta_en_consulta_per_Actual_total_fuerzas'] =Etiquetas_penultimo_paso['Llave_per'].isin(Consulta_recibos_per_Actual_varias_fuerzas['llave_orden_sql']).astype(int)

condicion_OTRA_fuerza=((Etiquetas_penultimo_paso['Esta_en_consulta_CONTACT']==0) & (Etiquetas_penultimo_paso['Esta_en_consulta_per_Actual_total_fuerzas']==1))
Etiquetas_penultimo_paso.loc[condicion_OTRA_fuerza, 'EstadoFinal'] = 'OTRA FUERZA DE VENTA_PER_ACT'

# VALIDACION FUERZA DE VENTA - POR PRIMER PERIODO (COD UNI)

In [18]:
"Nuevamente Divido mi liquidacion entre los que si obtuvieron su cod_uni y los que no"
Etiquetas_penultimo_paso_sin_cod_uni=[]
Etiquetas_penultimo_Con_cod_uni=[]
Etiquetas_penultimo_paso_sin_cod_uni=Etiquetas_penultimo_paso[Etiquetas_penultimo_paso['Llave_liq_cod_uni'].isna()]
Etiquetas_penultimo_Con_cod_uni=Etiquetas_penultimo_paso[Etiquetas_penultimo_paso['Llave_liq_cod_uni'].notna()]

In [19]:
# AQUI SOLO ARMO UNA CONSULTA MAS GLOBAL no por periodo SINO con los CODIGOS DE unidad (contemplando todos los periodos) con los programas
# de la liquidacion
Consulta_total_cod_uni=df_ordenes[df_ordenes['llave_total_cod_uni'].isin(Etiquetas_penultimo_Con_cod_uni['Llave_liq_cod_uni'])]

# Dejo en la consultA
Consulta_total_cod_uni = Consulta_total_cod_uni.sort_values(by=['llave_total_cod_uni', 'PERIODO'], ascending=[True, True])
#------------------EXPORTANDO EVIDENCIA-------------------------------------
from datetime import datetime
fecha_hoy_rc = datetime.now().strftime('%Y-%m-%d')
Consulta_total_cod_uni.to_excel(f'Evidencia_Recibos_cont_CODUNI{fecha_hoy_rc}_Reliq.xlsx')
#-------------------------------------------------
Consulta_total_cod_uni = Consulta_total_cod_uni.drop_duplicates(subset='llave_total_cod_uni', keep='first')
# Justo en este momento tengo la consulta lista con los primeros periodos
Consulta_solo_para_validar_nuevo_por_uni=Consulta_total_cod_uni.copy()
#-------------------------------------------------
Consulta_total_cod_uni=Consulta_total_cod_uni[Consulta_total_cod_uni['FuerzaComercialFinal'] != 'N/A']
Consulta_total_cod_uni=Consulta_total_cod_uni[Consulta_total_cod_uni['FuerzaComercialFinal'] != 'NA']
Consulta_total_cod_uni=Consulta_total_cod_uni[Consulta_total_cod_uni['FuerzaComercialFinal'] != 'PENDIENTE']
Consulta_total_cod_uni=Consulta_total_cod_uni[Consulta_total_cod_uni['FuerzaComercialFinal'].notna()]

#creo una consulta por codigo de unidad donde de verdad estan solo CONTACT
Consulta_recibos_per_COD_UNI_solo_CONTACT=Consulta_total_cod_uni[(Consulta_total_cod_uni['FuerzaComercialFinal'] == 'CONTACT')|(Consulta_total_cod_uni['FuerzaComercialFinal'] == 'CONTACT ')]

#AHORA SIMPLEMENTE COMPARO SI MIS LIQUIDACION ACTUAL EN SU RESPECTIVO PERIODO SI ES NUEVO
Etiquetas_penultimo_Con_cod_uni['Esta_en_consulta_CONTACT_CODUNI'] =Etiquetas_penultimo_Con_cod_uni['Llave_liq_cod_uni'].isin(Consulta_recibos_per_COD_UNI_solo_CONTACT['llave_total_cod_uni']).astype(int)
Etiquetas_penultimo_Con_cod_uni['Esta_en_consulta_total_fuerzas_CODUNI'] =Etiquetas_penultimo_Con_cod_uni['Llave_liq_cod_uni'].isin(Consulta_total_cod_uni['llave_total_cod_uni']).astype(int)

condicion_OTRA_fuerza_CODUNI=((Etiquetas_penultimo_Con_cod_uni['Esta_en_consulta_CONTACT_CODUNI']==0) & (Etiquetas_penultimo_Con_cod_uni['Esta_en_consulta_total_fuerzas_CODUNI']==1) &(~Etiquetas_penultimo_Con_cod_uni['TIPO'].str.contains('continuada', case=False, na=False)))
Etiquetas_penultimo_Con_cod_uni.loc[condicion_OTRA_fuerza_CODUNI, 'EstadoFinal'] = 'OTRA FUERZA DE VENTA_COD_U'
#--------------------------------------------------

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_6840\2116468290.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Etiquetas_penultimo_Con_cod_uni['Esta_en_consulta_CONTACT_CODUNI'] =Etiquetas_penultimo_Con_cod_uni['Llave_liq_cod_uni'].isin(Consulta_recibos_per_COD_UNI_solo_CONTACT['llave_total_cod_uni']).astype(int)
C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_6840\2116468290.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Etiquetas_penultimo_Con_cod_uni['Esta_en_consulta_tota

# VALIDACION = NUEVO POR PRIMER PERIODO Y COD_UNI

In [20]:
Consulta_solo_para_validar_nuevo_por_uni.rename(columns={'PERIODO': 'Primer_periodo_real'}, inplace=True)

In [21]:
#De la consulta sin duplicados por llave de codigo de unidad. Me traigo el primer periodo real a las etiquetas. (Aqui no se generan duplicados
# en las etiquetas)

Etiquetas_penultimo_Con_cod_uni = Etiquetas_penultimo_Con_cod_uni.merge(Consulta_solo_para_validar_nuevo_por_uni[['llave_total_cod_uni', 'Primer_periodo_real']], 
                                                left_on='Llave_liq_cod_uni', 
                                                right_on='llave_total_cod_uni',
                                                how='left')
Etiquetas_penultimo_Con_cod_uni.drop(columns=['llave_total_cod_uni'], inplace=True)
Etiquetas_penultimo_Con_cod_uni.shape
#----------------------------------------------
#----------------------------------------------

(48, 32)

In [22]:
#Si el periodo de la etiqueta no es igual al primer periodo real en el codigo de unidad =  ANTIGUO
condicion_antiguo_coduni=(
                (Etiquetas_penultimo_Con_cod_uni['Primer_periodo_real'].notna()) &
                (Etiquetas_penultimo_Con_cod_uni['PERIODO']!=Etiquetas_penultimo_Con_cod_uni['Primer_periodo_real']) &
                (~Etiquetas_penultimo_Con_cod_uni['TIPO'].str.contains('continuada', case=False, na=False)) 
)
Etiquetas_penultimo_Con_cod_uni.loc[condicion_antiguo_coduni, 'EstadoFinal'] = 'ANTIGUO_COD_U'

# Depuro columnas
Etiquetas_penultimo_Con_cod_uni.drop(columns=['Esta_en_consulta_Nuevos'], inplace=True)
Etiquetas_penultimo_Con_cod_uni.drop(columns=['Esta_en_consulta_per_Actual'], inplace=True)
Etiquetas_penultimo_Con_cod_uni.drop(columns=['Esta_en_consulta_CONTACT'], inplace=True)
Etiquetas_penultimo_Con_cod_uni.drop(columns=['Esta_en_consulta_per_Actual_total_fuerzas'], inplace=True)
Etiquetas_penultimo_Con_cod_uni.drop(columns=['Esta_en_consulta_CONTACT_CODUNI'], inplace=True)
Etiquetas_penultimo_Con_cod_uni.drop(columns=['Esta_en_consulta_total_fuerzas_CODUNI'], inplace=True)
Etiquetas_penultimo_Con_cod_uni.drop(columns=['Primer_periodo_real'], inplace=True)

Etiquetas_penultimo_Con_cod_uni.shape
#------------------------------------
#------------------------------------

(48, 25)

In [23]:
#VUELVO A UNIR MI LIQUIDACION
Etiquetas_penultimo_paso=[]
Etiquetas_penultimo_paso = pd.concat([Etiquetas_penultimo_Con_cod_uni, Etiquetas_penultimo_paso_sin_cod_uni], ignore_index=True)
#--------------------------------
#--------------------------------

# CRUZAR POR REFERIDOS

In [24]:
host2_3 = '172.16.1.33'
port2_3 = '1433'
database2_3 = 'CUN_REPOSITORIO'

try:
    url2_3 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_3, port=port2_3, db=database2_3)
    engine2_3 = create_engine(url2_3) 
    print("Conexion a la base de datos realizada")
    Lista_ref = pd.read_sql("select * from CUN_REPOSITORIO.Comisiones.HISTOR_REFERIDOS", engine2_3)
    Lista_ref2 = Lista_ref.copy()
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_3' in locals():
        engine2_3.dispose()
        print("Conexión cerrada")
Lista_ref

Conexion a la base de datos realizada
Conexión cerrada


,Listado_Referidos,Ejecucion
0,1000731646,_2024-10-23_
1,1047369554,_2024-10-23_
2,80016486,_2024-10-23_
3,1064193348,_2024-10-23_
4,1001115701,_2024-10-23_
...,...,...
3310,1088294078,_2026_10_02_
3311,1095821650,_2026_10_02_
3312,1103497398,_2026_10_02_
3313,1104261409,_2026_10_02_


In [25]:
Lista_ref=Lista_ref.astype(str)
Etiquetas_penultimo_paso['Refer'] = Etiquetas_penultimo_paso['Doc_Estudiante'].isin(Lista_ref['Listado_Referidos']).astype(int)
Etiquetas_penultimo_paso.loc[Etiquetas_penultimo_paso['Refer'] == 1, 'EstadoFinal'] = 'OK'

In [26]:
para_orden = Etiquetas_penultimo_paso['Doc_Estudiante'].unique()
valores_unicos_df1_orden = "', '".join(map(str, para_orden))
valores_unicos_df1_orden = f"'{valores_unicos_df1_orden}'"
periodos_unicos = Etiquetas_penultimo_paso['PERIODO'].unique()
valores_unicos_periodos = "', '".join(map(str, periodos_unicos))
valores_unicos_periodos = f"'{valores_unicos_periodos}'"

# CRUZAR POR REINTEGRO

In [27]:
##host2_1 = '172.16.1.33'
##port2_1 = '1433'
##database2_1 = 'CUN_REPOSITORIO'
##
##try:
##    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
##    engine2_1 = create_engine(url2_1) 
##    print("Conexion a la base de datos realizada")
##    cedulas_erroneas= pd.read_sql(f'''
##select * from CUN_REPOSITORIO.JUAN_GUZMAN.CEDULAS_NO_LIQUIDAR
##                            ''', engine2_1)
##    cedulas_erroneas_copia =  cedulas_erroneas.copy()
##    cedulas_erroneas3= pd.read_sql(f'''
##    SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal AS Fuerza_comercial
##FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS
##UNION ALL
##SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
##FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS_a
##UNION ALL
##SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
##FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS_25V02
##UNION ALL
##SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
##FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS_2025B
##UNION ALL
##SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
##FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS_25V02_s
##UNION ALL
##SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
##FROM CUN_REPOSITORIO.DBO.VENTAS_BASICA_MEDIA
##UNION ALL
##SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
##FROM CUN_REPOSITORIO.DBO.VENTAS25V02_contact
##UNION ALL
##SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
##FROM CUN_REPOSITORIO.DBO.VENTAS25V02_contact_2
##UNION ALL
##SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
##FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS_25V02_2
##UNION ALL
##SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
##FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS_2025B_2
##UNION ALL
##SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
##FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS_25V02_3
##UNION ALL
##SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
##FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS_25V02_4
##UNION ALL
##SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
##FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS_2025B_3;
##                            ''', engine2_1)
##        
##except Exception as e:
##    print("Error:", e)
##finally:
##    # Cerrar la conexión
##    if 'engine2_1' in locals():
##        engine2_1.dispose()
##        print("Conexión cerrada")
##cedulas_erroneas['Cliente']=cedulas_erroneas['Documento_Estudiante_zoho'].astype(float).astype(int).astype(str)
##'-----------------------------------------------------------------------------------------'
##'-----------------------------------------------------------------------------------------'
##cedulas_erroneas2 = pd.read_excel("EMORENO_Cedulas_reintegros_2025_ENERO.xlsx")
##cedulas_erroneas2['Cliente']=cedulas_erroneas2['Cliente'].astype(float).astype(int).astype(str)
##'-----------------------------------------------------------------------------------------'
##'-----------------------------------------------------------------------------------------'
##cedulas_erroneas = pd.concat([cedulas_erroneas, cedulas_erroneas2, cedulas_erroneas3], ignore_index=True)
##cedulas_erroneas = cedulas_erroneas.drop_duplicates()
##Etiquetas_penultimo_paso['Reint_Stv'] = Etiquetas_penultimo_paso['Doc_Estudiante'].isin(cedulas_erroneas['Cliente']).astype(int)
##Etiquetas_penultimo_paso.loc[Etiquetas_penultimo_paso['Reint_Stv'] == 1, 'EstadoFinal'] = 'OTRA_FUERZA_stv'
###---------------------------
###---------------------------

In [28]:
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'

try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    

    cedulas_erroneas3= pd.read_sql(f'''
    SELECT DOC_ALUM AS Documento_Estudiante_zoho, PERIODO, FUERZAZOHO AS Fuerza_comercial
    FROM CUN_REPOSITORIO.ZOHO.BASE_PERSONAS
    WHERE CLASE_ACTUAL = 'REINTEGRO'
    AND NUEVO = 'ANTIGUO'
    AND DOC_ALUM IN ({valores_unicos_df1_orden})
    AND PERIODO IN ({valores_unicos_periodos})
                            ''', engine2_1)
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")
cedulas_erroneas3['Cliente']=cedulas_erroneas3['Documento_Estudiante_zoho'].astype(float).astype(int).astype(str)
'-----------------------------------------------------------------------------------------'
'-----------------------------------------------------------------------------------------'
'-----------------------------------------------------------------------------------------'
'-----------------------------------------------------------------------------------------'
cedulas_erroneas = cedulas_erroneas3.copy()
cedulas_erroneas = cedulas_erroneas.drop_duplicates()


# Crear un set con las combinaciones válidas de cedulas y periodos
combos_invalidos = set(zip(cedulas_erroneas['Cliente'], cedulas_erroneas['PERIODO']))

# Aplicar validación combinada
Etiquetas_penultimo_paso['Reint_Stv'] = Etiquetas_penultimo_paso.apply(
    lambda row: int((row['Doc_Estudiante'], row['PERIODO']) in combos_invalidos), axis=1
)

Etiquetas_penultimo_paso.loc[Etiquetas_penultimo_paso['Reint_Stv'] == 1, 'EstadoFinal'] = 'REINTEGRO'

#---------------------------

Conexion a la base de datos realizada
Conexión cerrada


In [29]:
df_filtrado_1=Etiquetas_penultimo_paso.copy()

# VALIDACION - YA SE PAGO - MISMO CONTACT

In [30]:
host2_2 = '172.16.1.33'
port2_2 = '1433'
database2_2= 'CUN_REPOSITORIO'

try:
    url2_2 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_2, port=port2_2, db=database2_2)
    engine2_2= create_engine(url2_2) 
    print("Conexion a la base de datos realizada")
    archivo_contact = pd.read_sql(f'''
select * from CUN_REPOSITORIO.comisiones_contact.etiquetas where EstadoFinal='OK'
                                  ''', engine2_2)
    archivo_contact2 = archivo_contact.copy()
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_2' in locals():
        engine2_2.dispose()
        print("Conexión cerrada")
archivo_contact.shape

Conexion a la base de datos realizada
Conexión cerrada


(80469, 20)

In [31]:
#VALIDACION QUE SI FUNCIONO DEFINIR EL MES ARRIBA
# archivo_contact[archivo_contact['Periodo_Pago']==202501]

In [32]:
host2_4 = '172.16.1.33'
port2_4 = '1433'
database2_4= 'CUN_REPOSITORIO'

try:
    url2_4 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_4, port=port2_4, db=database2_4)
    engine2_4= create_engine(url2_4) 
    print("Conexion a la base de datos realizada")
    archivo_contact3 = pd.read_sql(f'''
select * from CUN_REPOSITORIO.comisiones_contact.EtiquetasReliquidacion where EstadoFinal='OK' and Periodo_Pago <>{MES_ACTUAL_EN_RELIQUIDACION}                                   
                                   ''', engine2_4)
    archivo_contact4 = archivo_contact3.copy()
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_4' in locals():
        engine2_4.dispose()
        print("Conexión cerrada")

Conexion a la base de datos realizada
Conexión cerrada


In [33]:
host2_4 = '172.16.1.33'
port2_4 = '1433'
database2_4= 'CUN_REPOSITORIO'

try:
    url2_4 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_4, port=port2_4, db=database2_4)
    engine2_4= create_engine(url2_4) 
    print("Conexion a la base de datos realizada")
    CONTACT_UNICD = pd.read_sql('''
select * from CUN_REPOSITORIO.comisiones_CONTACT.CONTACT_UNICD
WHERE COD_UNI IS NOT NULL
                                ''', engine2_4)
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_4' in locals():
        engine2_4.dispose()
        print("Conexión cerrada")

Conexion a la base de datos realizada
Conexión cerrada


ACA YA TENGO TODAS LAS ETIQUETAS OK (PAGADAS) - DE VENTAS NORMAL Y RELIQUIDACION

In [34]:
archivo_contact2 = pd.concat([archivo_contact2, archivo_contact4], ignore_index=True)
archivo_contact2['Doc_Estudiante']=archivo_contact2['Doc_Estudiante'].astype(str)

Solo me quedo de esas etiquetas OK solo las cedulas que estan en la liquidacion y me traigo el codigo unidad a esas etiquetas contact

In [35]:
archivo_contact2=archivo_contact2[archivo_contact2['Doc_Estudiante'].isin(df_filtrado_1['Doc_Estudiante'])]
archivo_contact2['Llave_etiq_cont']=archivo_contact2['Doc_Estudiante']+archivo_contact2['PERIODO']
# Quito duplicados de CONTACT_UNICD para añadir el cod_uni a las etiquetas
CONTACT_UNICD = CONTACT_UNICD.drop_duplicates(subset='llave_per', keep='first')

archivo_contact2 = archivo_contact2.merge(CONTACT_UNICD[['llave_per', 'COD_UNI']], 
                                     left_on='Llave_etiq_cont', 
                                     right_on='llave_per', 
                                     how='left')
archivo_contact2.rename(columns={'COD_UNI': 'COD_UNI_CONTACT'}, inplace=True)
archivo_contact2.drop(columns=['llave_per'], inplace=True)
#-----------------------------------------------
#-----------------------------------------------

VALIDACION VS HISTORICO CONTACT DIRECTO POR PERIODO

In [36]:
df_filtrado_1['Esta_en_hist_Cont_x_per']=df_filtrado_1['Llave_per'].isin(archivo_contact2['Llave_etiq_cont']).astype(int)
df_filtrado_1.loc[df_filtrado_1['Esta_en_hist_Cont_x_per']==1, 'EstadoFinal'] = 'YA SE PAGO'
#----------------------------------

VALIDACION VS HISTORICO CONCTACT POR CODIGO DE UNIDAD

In [37]:
"Nuevamente Divido mi liquidacion entre los que si obtuvieron su cod_uni y los que no"

df_final_sin_uni=df_filtrado_1[df_filtrado_1['Llave_liq_cod_uni'].isna()]
df_filtrado_1=df_filtrado_1[df_filtrado_1['Llave_liq_cod_uni'].notna()]

# ACA VALIDO PARA LOS QUE EN MI LIQ ACTUAL SI TIENEN CODIGO DE UNIDAD

Primero valido de mi liquidcion los que si tienen codigo de uidsad con una funcion que contemple dos cosas
- Los que en las etiquets si tienen cod uni
- Los que en las etiquetas no tiene cod uni

In [38]:
def comparar_y_agregar_pert_contact(df_final, archivo_contact2):
    # Iterar sobre cada fila de df_final
    for idx, row in df_final.iterrows():
        # Obtener el valor de 'DOC_ALUM', 'COD_UNI' y 'EstadoFinal' de la fila actual de df_final
        doc_alum = row['Doc_Estudiante']
        cod_uni = row['COD_UNI']
        estado_fini = row['EstadoFinal']
        
        # Buscar en archivo_contact2 si existe alguna fila que tenga 'Doc_Estudiante' igual a 'DOC_ALUM'
        match = archivo_contact2[archivo_contact2['Doc_Estudiante'] == doc_alum]
        
        # Si no se encuentra coincidencia en 'Doc_Estudiante'
        if match.empty:
            df_final.at[idx, 'EstadoFinal'] = estado_fini
        else:
            # Si se encuentra coincidencia, comprobar la columna 'COD_UNI_CONTACT'
            cod_uni_contact = match['COD_UNI_CONTACT'].values[0]
            
            if pd.isna(cod_uni_contact):  # Si 'COD_UNI_CONTACT' es NaN
                df_final.at[idx, 'EstadoFinal'] = 'YA SE PAGO'
            elif cod_uni == cod_uni_contact:  # Si 'COD_UNI' coincide con 'COD_UNI_CONTACT'
                df_final.at[idx, 'EstadoFinal'] = 'YA SE PAGO'
            else:  # Si no coincide
                df_final.at[idx, 'EstadoFinal'] = estado_fini
    
    return df_final
#--------------------------------------------------------

In [39]:
df_filtrado_1 = comparar_y_agregar_pert_contact(df_filtrado_1, archivo_contact2)
#------------------------------------

# ACA VALIDO PARA LOS QUE EN MI LIQ ACTUAL NO TIENEN CODIGO DE UNIDAD

In [40]:
df_final_sin_uni['Esta_en_hist_cont'] = df_final_sin_uni['Doc_Estudiante'].isin(archivo_contact2['Doc_Estudiante']).astype(int)
df_final_sin_uni.loc[df_final_sin_uni['Esta_en_hist_cont'] == 1, 'EstadoFinal'] = 'YA SE PAGO'

Uno nuevamente mi liquidacion que habia dividido en dos partes

In [41]:
df_filtrado_1 = pd.concat([df_filtrado_1, df_final_sin_uni], ignore_index=True)

Preparo la liq actual para inyectarla en CONTAC_UNICD

In [42]:
df_filtrado_1.rename(columns={'Llave_per': 'llave_per'}, inplace=True)
CONTAC_UNI_TEMP=df_filtrado_1.copy()
CONTAC_UNI_TEMP['Fecha_de_pago_Real']="-"
CONTAC_UNI_TEMP['Fecha_de_pago']="-"
CONTAC_UNI_TEMP=CONTAC_UNI_TEMP[['Doc_Estudiante',	'PERIODO',	'Documento_del_Asesor',	'Fecha_de_pago_Real',	'Propietario_del_registro',	
                             'Programa_vendido',	'Fecha_de_pago',	'Correo_Asesor',	'Periodo_Pago',	'EST_MATRICULADO',	
                             'NCR_ANULACION',	'TIPO',	'ORDEN',	'VAL_LIQUIDADO',	'NUEVO',	'ESTADO_PAGO',	'Documento_Estudiante_zoho',
                             'Motivo',	'Propietario_de_Fase_Name',	'EstadoFinal',	'llave_per',	'COD_UNI']]
#-----------------------

In [43]:
CONTAC_UNI_TEMP.shape

(49, 22)

In [44]:
host_2_12 = '172.16.1.33'
port_2_12 = '1433'
database_2_12 = 'CUN_REPOSITORIO'
url_2_12 = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host_2_12, port=port_2_12, db=database_2_12)
engine_2_12 = create_engine(url_2_12, fast_executemany=True)

table_name_2_12 = 'CONTACT_UNICD'
schema_name_2_12 = 'COMISIONES_CONTACT'
#Send the data to sql server
CONTAC_UNI_TEMP.to_sql(
     name=table_name_2_12,
     schema=schema_name_2_12,
     con=engine_2_12,
     if_exists='append',
     index=False,
     chunksize = 10000
 )

# # End connection
engine_2_12.dispose()

In [45]:
#VALIDADO - SI TIENEN LAS MISMAS COLUMNAS PARA INYECCION, MISMO ORDEN Y MISMO TIPO

# dtypes = df_filtrado_1.dtypes
# dtypes_df = pd.DataFrame(dtypes).reset_index()
# dtypes_df.columns = ['Columna', 'Tipo de Dato']
# dtypes_df#.to_excel('tipos_de_dato_final_base_pago.xlsx', index=False)
# # Tabla_Conteos_SP.dtypes

Preparacion para inyectar en TEMP_ETIQ_ACTUAL_1_1

In [46]:
df_filtrado_1=df_filtrado_1[['Doc_Estudiante',	'PERIODO',	'Documento_del_Asesor',	'Propietario_del_registro',	'Programa_vendido',	
                             'Correo_Asesor',	'Periodo_Pago',	'EST_MATRICULADO',	'NCR_ANULACION',	'TIPO',	'ORDEN',	'VAL_LIQUIDADO',	
                             'NUEVO',	'ESTADO_PAGO',	'Documento_Estudiante_zoho',	'Motivo',	'Propietario_de_Fase_Name',	'EstadoFinal'
                            ]]
#--------------------
#--------------------

In [47]:
df_filtrado_1.shape

(49, 18)

In [48]:
host_2_12 = '172.16.1.33'
port_2_12 = '1433'
database_2_12 = 'CUN_REPOSITORIO'
url_2_12 = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host_2_12, port=port_2_12, db=database_2_12)
engine_2_12 = create_engine(url_2_12, fast_executemany=True)

table_name_2_12 = 'EtiquetasReliquidacion'
schema_name_2_12 = 'COMISIONES_CONTACT'
# # Send the data to sql server
df_filtrado_1.to_sql(
     name=table_name_2_12,
     schema=schema_name_2_12,
     con=engine_2_12,
     if_exists='append',
     index=False,
     chunksize = 10000
 )

# # End connection
engine_2_12.dispose()

In [49]:
#CONSULTA PARA CREAR LA TABLA TEMPORAL EN SQL 
# Busca en tabla Base pago mes anterior, y cruza contra las reliquidadas para sacar la diferencia 
with engine.connect() as connection:
    try:

        sql_query = text(f'''
        IF object_id('CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_BaseReliquidacionCal') IS NOT NULL
                DROP TABLE CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_BaseReliquidacionCal

            Select

                distinct t1.*,
                        isnull(t2.Pregrados_Reliquidacion,0)Pregrados_Reliquidacion,
                        isnull(t2.Postgrado_Reliquidacion,0)Postgrado_Reliquidacion,
                        isnull(t2.Educacion_continuada_Reliquidacion,0)Educacion_continuada_Reliquidacion
            INTO CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_BaseReliquidacionCal
                from Comisiones_Contact.BasePago t1
                left join (select distinct Documento_del_Asesor,Periodo_Pago,
                sum(case when TIPO = 'Pregrado' then 1 else 0 end) As Pregrados_Reliquidacion,
                sum(case when TIPO = 'Postgrado' then 1 else 0 end) As Postgrado_Reliquidacion,
                sum(case when TIPO = 'Educacion continuada' then 1 else 0 end) As Educacion_continuada_Reliquidacion
                from Comisiones_Contact.EtiquetasReliquidacion where EstadoFinal = 'OK'
                                                                and Periodo_Pago = {MES_ACTUAL_EN_RELIQUIDACION}
                group by Documento_del_Asesor,Periodo_Pago) t2
                                                        on t1.Documento_del_Asesor = t2.Documento_del_Asesor
                where t1.Periodo_Pago = {MES_ACTUAL_EN_RELIQUIDACION}

        ''')
        connection.execute(sql_query)
        connection.commit()  # Asegurar que los cambios se guarden en la base de datos
        print("Consulta realizada con exito")
    except Exception as e:
        print(f"Error al eliminar registros: {e}")
    finally:
        connection.close()

Consulta realizada con exito


In [50]:
##Realiza los calculos de la liquidacion del mes pasado contra reliquidacion, carga en tabla BasePagoreliquidaciones 
with engine.connect() as connection:
    try:

        sql_query = text(f'''
        Delete from Comisiones_Contact.BasePagoReliquidaciones where Periodo_Pago = {MES_ACTUAL_EN_RELIQUIDACION}
        delete from Comisiones_Contact.RELIQ_PREGR_APROB_CONTAC_MOD2 where Periodo_Pago_1 = {MES_ACTUAL_EN_RELIQUIDACION}
        delete from Comisiones_Contact.RELIQ_SOL_PREG_X where Periodo_Pago_1 = {MES_ACTUAL_EN_RELIQUIDACION}
        delete from Comisiones_Contact.RELIQ_DESGLO_BASE where Per_desglo = {MES_ACTUAL_EN_RELIQUIDACION}
          
        ''')
        connection.execute(sql_query)
        connection.commit()  # Asegurar que los cambios se guarden en la base de datos
        print("Consulta realizada con exito")
    except Exception as e:
        print(f"Error al eliminar registros: {e}")
    finally:
        connection.close()

Consulta realizada con exito
